In [1]:
import numpy as np

from geotiff.geotiff import ReadGeoTiff, WriteGeoTiff
from percolation.percolation import Percolation

In [2]:
filename = 'datasets/processed/gebco_2026_sub_ice_geotiff.tif'

_info, _arr = ReadGeoTiff(filename)

/Users/erneson/unifor/blur/.venv/lib/python3.13/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [3]:
k = 100
_arr = _arr[::k, ::k]

h_star = 0
p = _arr > h_star

In [4]:
percolation = Percolation(p)
percolation.site_percolation()

label = percolation.parent.reshape(percolation.nrows, percolation.ncols)

vals, sizes = np.unique(label[p], return_counts = True)

432 864 373248


/Users/erneson/unifor/blur/percolation/percolation.py:24: RuntimeWarning: overflow encountered in scalar add
  self.rank[y] += 1


In [5]:
label.nbytes / 1e9

0.002985984

In [6]:
vals

array([ 14929,  16802,  17493,  17863,  18711,  18721,  18806,  19554,
        19560,  19571,  19589,  19592,  19631,  20352,  20448,  20451,
        20929,  21290,  21311,  22216,  22644,  22928,  23141,  23525,
        23705,  23786,  23824,  24359,  24544,  25213,  26073,  26100,
        26135,  26403,  26550,  27429,  27830,  27838,  27850,  28141,
        28240,  28438,  28668,  28700,  29524,  29617,  30140,  30422,
        30445,  30532,  30869,  30998,  31017,  31024,  31027,  31345,
        32166,  32265,  32311,  32591,  32604,  32606,  33128,  33602,
        33887,  34468,  34892,  35162,  35611,  35722,  36035,  36046,
        37943,  38347,  38881,  40908,  40912,  40935,  40941,  41741,
        42046,  42308,  42550,  42580,  42660,  42663,  42810,  42942,
        43373,  43443,  43447,  43670,  43673,  43749,  44255,  44383,
        44532,  45177,  45249,  45395,  46016,  46255,  47534,  49642,
        51204,  52358,  54452,  54766,  55321,  55541,  55624,  56395,
      

In [7]:
sizes

array([  778,     1,     3,     4,    14,     1,    69,     1,     8,
           5,     2,     6,     3,    39,     1,     2,     3,     1,
           2,     1,    30,   106,    25,    21,     1,     1,     1,
          21,     2,     3,    29,     2,     3,     1,     1,     1,
           1,     1,    10,     1,   147,     2,    84,    36,     3,
           1,    31,     1,    14,     1,     1,     1,     9,     2,
          10,     1,     1,     1,     1,     1,     1,     1,     2,
           2,    51,     9,     1,     3,     2,     2,     1,     2,
           1,     5,     8,     1,    18,     3,     4,     5,     4,
           1,     1,     1,     2,     1,     1,     1,   433,     1,
           1,     1,     1,     7,     1,     1,     1,    13,     1,
           2,     1,     1,     1,   107,    51,     1,     1,     1,
           3,     2,  2972,     7,     2,     1,     2,     1,     1,
           1,     1,     2,     3,     2,     2,     1,     1,     1,
           2,     1,

In [8]:
nrows, ncols = label.shape

for val in vals:
    # arr
    cluster = label == val
    
    empty_cols = np.flatnonzero(~cluster.any(axis = 0))
    if empty_cols.size:
        i0 = empty_cols[0]
    else:
        i0 = 0
    
    ys, xs = np.nonzero(cluster)
    xs = (xs - i0) % ncols
    
    j_min, j_max = ys.min(), ys.max()
    i_min, i_max = xs.min(), xs.max()
    cols = (np.arange(i_min, i_max + 1) + i0) % ncols

    cluster_cropped = cluster[j_min : j_max + 1, cols]
    _arr_cropped = _arr[j_min:j_max + 1, cols]

    arr = np.where(cluster_cropped,
                   _arr_cropped,
                   _info['NoDataValue'])
    # arr

    # info
    height, width = arr.shape

    _gt = _info['GeoTransform']
    y_line = i_min
    x_pixel = cols[0]
    
    x_geo = _gt[0] + x_pixel * _gt[1] + y_line * _gt[2]
    y_geo= _gt[3] + x_pixel * _gt[4] + y_line * _gt[5]

    gt = (x_geo, _gt[1], _gt[2], y_geo, _gt[4], _gt[5])

    info = {}
    info['RasterYSize'] = height
    info['RasterXSize'] = width
    info['GeoTransform'] = gt
    info['Projection'] = _info['Projection']
    info['NoDataValue'] = _info['NoDataValue']
    # info
    
    filename = 'datasets/processed/land/gebco_2026_sub_ice_geotiff_{}.tif'.format(val)
    
    WriteGeoTiff(info,
                 arr,
                 filename)